In [1]:
path = 'train_images_mapped/'


In [2]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing import image
import os
from tqdm import tqdm
import tensorflow as tf

# Clear any cached models to avoid conflicts
tf.keras.backend.clear_session()

from tensorflow.keras.applications.efficientnet import EfficientNetB3, preprocess_input

# Use the correct input size for EfficientNetB3 (300x300, not 224x224)
# And explicitly specify input_shape to ensure 3 channels
base_model = EfficientNetB3(
    weights='imagenet', 
    include_top=False, 
    pooling='avg',
    input_shape=(300, 300, 3)  # Explicitly specify 3 channels
)
# --- 2. Load your DataFrames ---
image_folder = path
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv') # Load test.csv too

# --- 3. Feature Extraction Function ---
# It's good practice to wrap this in a function so you can reuse it for the test set

def extract_features(df, img_folder):
    extracted_features = []
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):

        # ✅ THIS IS THE KEY LINE ✅
        # Use the 'sample_id' from the row to build the correct file path.
        img_path = os.path.join(img_folder, str(row['sample_id']) + '.jpg')

        try:
            # Use 300x300 for EfficientNetB3 (not 224x224)
            img = image.load_img(img_path, target_size=(300, 300))
            img_array = image.img_to_array(img)
            expanded_img_array = np.expand_dims(img_array, axis=0)
            preprocessed_img = preprocess_input(expanded_img_array)

            features = base_model.predict(preprocessed_img, verbose=0)
            extracted_features.append(features.flatten())

        except FileNotFoundError:
            # Important: If an image is missing, append a placeholder (like zeros).
            # This ensures your feature array stays perfectly aligned with the DataFrame.
            # EfficientNetB3 with pooling='avg' outputs 1536 features, not 2048
            extracted_features.append(np.zeros(1536))

    return np.array(extracted_features)

# --- 4. Run the Extraction for Both Train and Test Sets ---

print("Extracting features for the training set...")
train_features = extract_features(train_df, image_folder)

# print("\nExtracting features for the test set...")
# test_features = extract_features(test_df, image_folder)

# --- 5. Save the Features ---
np.save('train_resnet50_features_efficientnet.npy', train_features)
# np.save('test_resnet50_features.npy', test_features)

print(f"\nDone. Train features shape: {train_features.shape}")
# print(f"Done. Test features shape: {test_features.shape}")

ValueError: Shape mismatch in layer #1 (named stem_conv)for weight stem_conv/kernel. Weight expects shape (3, 3, 1, 40). Received saved weight with shape (3, 3, 3, 40)

In [ ]:
# Verify the model is working correctly
print("=== VERIFYING MODEL SETUP ===")
print(f"Base model input shape: {base_model.input_shape}")
print(f"Base model output shape: {base_model.output_shape}")

# Test with a dummy image to verify everything works
print("\n=== TESTING WITH DUMMY IMAGE ===")
dummy_img = np.random.rand(1, 300, 300, 3) * 255  # Random image
dummy_preprocessed = preprocess_input(dummy_img)
try:
    dummy_features = base_model.predict(dummy_preprocessed, verbose=0)
    print(f"✅ Model works! Output shape: {dummy_features.shape}")
    print(f"Feature vector size: {dummy_features.flatten().shape[0]}")
except Exception as e:
    print(f"❌ Error: {e}")

# Check if image folder exists and has images
print(f"\n=== CHECKING IMAGE FOLDER ===")
print(f"Image folder: {image_folder}")
print(f"Folder exists: {os.path.exists(image_folder)}")

if os.path.exists(image_folder):
    images = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]
    print(f"Number of .jpg files: {len(images)}")
    if images:
        print(f"Sample files: {images[:5]}")
        
        # Test loading one actual image
        test_img_path = os.path.join(image_folder, images[0])
        try:
            test_img = image.load_img(test_img_path, target_size=(300, 300))
            print(f"✅ Successfully loaded test image: {images[0]}")
            print(f"Image size after loading: {test_img.size}")
        except Exception as e:
            print(f"❌ Error loading test image: {e}")

print("\n=== READY FOR FEATURE EXTRACTION ===")
print("If all checks passed, you can proceed with feature extraction.")